# Vision Transformers

#Steps:
Patch extraction

1.   Patch embedding
2. CLS token
3. Positional embeddings
4. Single-head attention first
5. Multi-head attention
6. MLP block
7. One Transformer encoder block
8. Stack 12 encoder blocks
9. Classification head
10. Training loop
11. Compare with 97% CNN


##1. patch extarction

*   load the image
*   make the patches for every image
the size of the image patches like that
[channels,patch rows,patch rows, patch columns, patch height , patch width]
*   flatten the patches so they are like the transformers tokens







#2. CLS token
CLS token as a learned summary representation of the whole image
it has the same size as the other tokens the features that will be in it will the most important ones that we get from the other tokens so . it contains information about all the other tokens

##3. positional embedding:
in here the positional embeddings are the position informatin that the transformer wants to know the position of each token the original transformer they provided as give not learnable but in vit they thought that lets make it a learnable as maybe the model think this representation are not good for the model

## 4. Single-head attention
the idea behind it is that for all the patches i have to make their q ,k and value vector but at this point we need to just make it as learnable parameter

define a class for vision transformer

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/dataset.zip" -d dataset

encoder block steps:
1. we start by normalising the input
2. then we initialize the q k and v
3. perform multihead attention

In [3]:
from torch import nn
class EncoderBlock(nn.Module):
  def __init__(self,embed_dim=512,head_nums= 8):
    super().__init__()
    self.embed_dim = embed_dim
    self.head_nums = head_nums
    self.WQ = nn.Linear(self.embed_dim,self.embed_dim)
    self.WK = nn.Linear(self.embed_dim,self.embed_dim)
    self.WV = nn.Linear(self.embed_dim,self.embed_dim)
    self.WO = nn.Linear(self.embed_dim,self.embed_dim)
    self.layer_norm1 = nn.LayerNorm(self.embed_dim)
    self.layer_norm2 = nn.LayerNorm(self.embed_dim)
    self.attention_dropout = nn.Dropout(0.1)
    self.mlp = nn.Sequential(
        nn.Linear(self.embed_dim, self.embed_dim*4),
        nn.Dropout(0.1),
        nn.GELU(),
        nn.Linear(self.embed_dim*4, self.embed_dim),
        nn.Dropout(0.1))

  def forward(self,x):
    # layer norm
    B = x.shape[0]
    patch_nums = x.shape[1]
    residual = x
    patches = self.layer_norm1(x)
    # multi head attention
    Q = self.WQ(patches)
    K = self.WK(patches)
    V = self.WV(patches)
    # Multi-head attention splits the Q, K, and V feature dimensions into multiple heads
    # so that the model can learn multiple attention relationships between the patches in parallel.
    Q = Q.reshape(B,patch_nums,self.head_nums,-1)
    Q = Q.permute(0,2,1,3)
    K = K.reshape(B,patch_nums,self.head_nums,-1)
    K = K.permute(0,2,1,3)
    V = V.reshape(B,patch_nums,self.head_nums,-1)
    V = V.permute(0,2,1,3)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    # scores[32, 8, 197,197]
    # the next step for scaling the score so we are avoinding the gradient problems the sqrt of this quantity is equal to the std of the score
    attention = scores/math.sqrt(self.embed_dim//self.head_nums)
    attention = torch.softmax(attention,dim=-1)
    attention = self.attention_dropout(attention)
    output = torch.matmul(attention,V)
    # [32, 8, 197,197] [32,8,197,64]= [32,8,197,64]
    output = output.permute(0,2,1,3)
    # [32, 197, 8,64]

    # collecting the features again together
    output = output.reshape(B,patch_nums,-1)
    # [32, 197, 512]
    # the next step is to let the features that are comming from all the different heads interact
    output = self.WO(output)

    # residual connection
    patches =  residual + output
    residual = patches
    # layer norm
    patches = self.layer_norm2(patches)
    # mlp
    patches = self.mlp(patches)
    # residual connection
    patches = residual + patches
    return patches

##steps:
1. extract patch the shape would be [32,196,786]
2. get the embedding [32,196,512 ]:the embeddings are learning parameters that learned during training the dimension can be hyperparametrs
3. cls token which is a hyperparameter
4. the pos whick is learning parametr too
5. the encoder block whch is been repeated 6 times based on the vit paper


In [4]:
from torch import nn
import math
class VisionTransformer(nn.Module):
  def __init__(self,patch_size = 16,channels = 3,embed_dim = 512, image_size=224,head_nums=8):
    super().__init__()
    self.patch_size = patch_size
    self.head_nums = head_nums
    self.patch_dim = channels * patch_size * patch_size
    self.patch_nums = (image_size//patch_size)**2
    self.embed_dim = embed_dim
    self.patch_embeddings= nn.Linear(self.patch_dim , self.embed_dim )
    self.cls = nn.Parameter(torch.randn(1,1,self.embed_dim))
    self.POS = nn.Parameter(torch.randn(1,self.patch_nums+1,self.embed_dim))
    self.classifier = nn.Linear(self.embed_dim,8)
    self.encoders = nn.ModuleList([
    EncoderBlock(embed_dim= self.embed_dim,head_nums=self.head_nums) for _ in range(6)])

  def make_patches(self,images):
    """ take the image and patch it pased on the patch size and return the flatten patches """
    B, C, H, W = images.shape
    patches = images.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
    patches = patches.permute(0,2,3,1,4,5)
    patches = patches.reshape(B, self.patch_nums,-1)
    return patches

  def forward(self,x):
    B,C,H,W = x.shape
    # 1.
    patches = self.make_patches(x)
    # 2.
    patches_embbedding = self.patch_embeddings(patches)
    #3.
    cls = self.cls.expand(B,-1,-1)
    all_patches = torch.cat((cls,patches_embbedding),dim=1)
    #4
    pos = self.POS.expand(B,-1,-1)
    all_patches = all_patches + pos
    for encoder in self.encoders:
      all_patches = encoder(all_patches)
    cls_token = all_patches[:,0]
    logits = self.classifier(cls_token)
    return logits


In [5]:
from PIL import Image
from torchvision.datasets import ImageFolder
from torchvision import transforms, torch
from torch.utils.data import random_split
from torch.utils.data import DataLoader
transform = transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor(),])
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
train_dataset = ImageFolder(root="dataset/dataset/train",transform=transform)
val_dataset = ImageFolder(root="dataset/dataset/val",transform=transform)
test_dataset = ImageFolder(root="dataset/dataset/test", transform=transform)

In [6]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [7]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 20
model = VisionTransformer(patch_size=16).to(device)
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)
loss_fn = torch.nn.CrossEntropyLoss()

In [8]:
import torch

print(torch.cuda.is_available())
print(next(model.parameters()).device)

True
cuda:0


In [9]:
def train_epoch(train_loader,los_fn,optimizer ):
  model.train()
  train_correct = 0
  train_total = 0
  train_loss = 0.0
  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    pred = model(images)
    loss = loss_fn(pred,labels)
    predicted_classes = torch.argmax(pred, dim=1)
    train_correct += (predicted_classes == labels).sum().item()
    train_total += labels.size(0)
    train_loss += loss.item()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0)
    optimizer.step()
  train_accuracy = train_correct / train_total
  train_loss /= len(train_loader)
  return train_loss,train_accuracy


In [10]:
def validate_epoch(val_loader,loss_fn, optimizer):
  model.eval()
  val_correct = 0
  val_total = 0
  val_loss= 0.0
  with torch.no_grad():
      for images, labels in val_loader:
          images = images.to(device)
          labels = labels.to(device)
          outputs = model(images)
          loss = loss_fn(outputs, labels)
          predictions = outputs.argmax(dim=1)
          val_loss += loss.item()
          val_correct += (predictions == labels).sum().item()
          val_total += labels.size(0)
  val_accuracy = val_correct / val_total
  val_loss /= len(val_loader)
  return val_loss, val_accuracy

In [11]:

for epoch in range (epochs):
  train_loss,train_accuracy = train_epoch(train_loader,loss_fn,optimizer)
  current_lr = optimizer.param_groups[0]['lr']
  val_loss,val_accuracy = validate_epoch(val_loader,loss_fn,optimizer)
  scheduler.step(val_loss)
  print(
        f"Epoch {epoch+1}/{ epochs} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}|"
        f"Val Loss: {val_loss:.4f}|"
        f"LR: {current_lr:.2e}")

Epoch 1/20 | Train Acc: 0.3510 | Train Loss: 1.7097 | Val Acc: 0.4750|Val Loss: 1.3942|LR: 1.00e-04
Epoch 2/20 | Train Acc: 0.6459 | Train Loss: 0.9560 | Val Acc: 0.7463|Val Loss: 0.6716|LR: 1.00e-04
Epoch 3/20 | Train Acc: 0.7695 | Train Loss: 0.6222 | Val Acc: 0.8194|Val Loss: 0.4702|LR: 1.00e-04
Epoch 4/20 | Train Acc: 0.8470 | Train Loss: 0.4206 | Val Acc: 0.8574|Val Loss: 0.3827|LR: 1.00e-04
Epoch 5/20 | Train Acc: 0.8946 | Train Loss: 0.3027 | Val Acc: 0.8824|Val Loss: 0.3322|LR: 1.00e-04
Epoch 6/20 | Train Acc: 0.9072 | Train Loss: 0.2675 | Val Acc: 0.9194|Val Loss: 0.2457|LR: 1.00e-04
Epoch 7/20 | Train Acc: 0.9224 | Train Loss: 0.2327 | Val Acc: 0.8917|Val Loss: 0.2845|LR: 1.00e-04
Epoch 8/20 | Train Acc: 0.9328 | Train Loss: 0.1951 | Val Acc: 0.9306|Val Loss: 0.1699|LR: 1.00e-04
Epoch 9/20 | Train Acc: 0.9409 | Train Loss: 0.1715 | Val Acc: 0.9407|Val Loss: 0.1748|LR: 1.00e-04
Epoch 10/20 | Train Acc: 0.9488 | Train Loss: 0.1445 | Val Acc: 0.9102|Val Loss: 0.3118|LR: 1.00e-04

In [12]:
def test(test_loader):
  model.eval()
  test_correct = 0
  test_total = 0
  test_loss= 0.0
  with torch.no_grad():
      for images, labels in test_loader:
          images = images.to(device)
          labels = labels.to(device)
          outputs = model(images)
          loss = loss_fn(outputs, labels)
          predictions = outputs.argmax(dim=1)
          test_loss += loss.item()
          test_correct += (predictions == labels).sum().item()
          test_total += labels.size(0)
  test_accuracy = test_correct / test_total
  test_loss /= len(test_loader)
  print(
        f"test Acc: {test_accuracy:.4f}|"
        f"test Loss: {test_loss:.4f}|"
    )
test(test_loader)

test Acc: 0.9725|test Loss: 0.1354|


In [13]:
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Total parameters: 19413512


In [14]:
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}, 'vit_checkpoint.pth')

NameError: name 'loss' is not defined